# Stock Close Price Forecasting

## Project Overview

Forecasts a **daily stock closing price** (USD) over a 14-day horizon. Synthetic data spans 2 years with geometric Brownian motion, earnings jumps, and volatility regimes.

**Models**: Naive, LazyPredict, FLAML, StatsForecast. **Horizon**: 14 days.

## Learning Objectives

1. Understand time-series patterns (trend, seasonality, noise)
2. Build naive and seasonal-naive baselines
3. Engineer lag and rolling features for tabular ML
4. Use LazyPredict for quick regression benchmarking
5. Apply FLAML AutoML (excluding XGBoost due to compatibility)
6. Use StatsForecast (AutoARIMA, AutoETS, SeasonalNaive)
7. Compare all approaches with MAE / RMSE / MAPE

## Problem Statement

Given historical daily stock closing prices, predict the next 14 days. Stock price forecasting teaches the challenge of predicting near-efficient markets and the importance of proper evaluation methodology.

## Why This Project Matters

Stock markets are the world's largest prediction challenge. Billions of dollars in quantitative trading research attempt to forecast prices. This project teaches why naive benchmarks are surprisingly hard to beat, and why evaluation methodology matters more than model complexity.

## Dataset Overview

Synthetic dataset:
- 730 days (2 years) of daily stock closing prices
- Geometric Brownian Motion with drift
- Quarterly earnings surprise jumps
- Volatility regimes (calm and turbulent)
- No weekly seasonality (but weekday-only trading approximated)

| Column | Description |
|--------|-------------|
| `date` | Date |
| `close_usd` | Daily stock closing price (USD) |

## Dataset Source and License Notes

Synthetically generated for educational purposes. No external download required.

## Environment Setup

In [1]:
import subprocess, importlib
for pkg in ['numpy', 'pandas', 'matplotlib', 'scikit-learn', 'statsforecast', 'flaml', 'lazypredict']:
    try:
        importlib.import_module(pkg.replace('-', '_').split('[')[0])
    except ImportError:
        subprocess.check_call(['pip', 'install', '-q', pkg])
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## Imports

In [2]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import mean_absolute_error, mean_squared_error
print("Imports complete.")

Imports complete.


## Configuration / Constants

In [3]:
SEED = 42
N_POINTS = 730
HORIZON = 14
VAL_SIZE = 14
TEST_SIZE = 14
SEASON_LENGTH = 7
FREQ = 'D'
TARGET = 'close_usd'
np.random.seed(SEED)
print(f"Config: {N_POINTS} points, horizon={HORIZON}, season={SEASON_LENGTH}")

Config: 730 points, horizon=14, season=7


## Dataset Generation

In [4]:
dates = pd.date_range(start='2022-01-01', periods=N_POINTS, freq='D')
t = np.arange(N_POINTS)

# Geometric Brownian Motion
mu_daily = 0.0003  # ~8% annual return
sigma_daily = 0.015  # ~24% annual volatility

log_price = np.zeros(N_POINTS)
log_price[0] = np.log(150)  # starting price

# Volatility regimes
vol_mult = np.ones(N_POINTS)
for s in range(0, N_POINTS, 200):
    high_start = min(s + np.random.randint(80, 160), N_POINTS - 30)
    dur = min(np.random.randint(20, 50), N_POINTS - high_start)
    vol_mult[high_start:high_start + dur] = 2.0

for i in range(1, N_POINTS):
    log_price[i] = log_price[i-1] + mu_daily + sigma_daily * vol_mult[i] * np.random.normal()

# Quarterly earnings jumps
for q in range(0, N_POINTS, 90):
    earn_day = min(q + np.random.randint(85, 95), N_POINTS - 1)
    jump = np.random.normal(0, 0.03)  # 3% surprise std
    log_price[earn_day:] += jump

close_usd = np.exp(log_price).round(2)

df = pd.DataFrame({'date': dates, 'close_usd': close_usd})
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (730, 2)


,date,close_usd
0,2022-01-01,150.00
1,2022-01-02,148.00
2,2022-01-03,147.76
3,2022-01-04,143.42
4,2022-01-05,142.40


## Data Validation Checks

In [5]:
assert df.isnull().sum().sum() == 0, "Missing values"
assert (df[TARGET] > 0).all(), "Non-positive values"
assert df['date'].is_monotonic_increasing, "Not sorted"
assert len(df) == N_POINTS, "Row count mismatch"
print("All validation checks passed.")

All validation checks passed.


## Exploratory Data Analysis

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0, 0].plot(df['date'], df[TARGET], linewidth=0.6)
axes[0, 0].set_title('close_usd Over Time')
axes[0, 1].hist(df[TARGET], bins=30, edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Distribution')
from pandas.plotting import autocorrelation_plot
autocorrelation_plot(df[TARGET], ax=axes[1, 0])
axes[1, 0].set_xlim(0, min(60, N_POINTS // 2))
axes[1, 0].set_title('Autocorrelation')
rw = min(SEASON_LENGTH * 2, N_POINTS // 4)
axes[1, 1].plot(df['date'], df[TARGET].rolling(rw).mean())
axes[1, 1].set_title(f'Rolling {rw}-period Mean')
plt.tight_layout()
plt.savefig('eda.png', dpi=100, bbox_inches='tight')
plt.show()
print("EDA complete.")

EDA complete.


## Target Analysis

In [7]:
print("close_usd Statistics:")
print(df[TARGET].describe())
print(f"\nCV: {df[TARGET].std() / df[TARGET].mean():.3f}")
print(f"Skewness: {df[TARGET].skew():.3f}")

close_usd Statistics:
count    730.000000
mean     146.484685
std       14.142116
min      117.320000
25%      135.930000
50%      145.170000
75%      154.847500
max      189.000000
Name: close_usd, dtype: float64

CV: 0.097
Skewness: 0.458


## Train / Validation / Test Split Strategy

Time-aware split (no shuffling):
- **Train**: all except last 28
- **Validation**: 14 points
- **Test**: last 14 points

In [8]:
train = df.iloc[:-(VAL_SIZE + TEST_SIZE)].copy()
val = df.iloc[-(VAL_SIZE + TEST_SIZE):-TEST_SIZE].copy()
test = df.iloc[-TEST_SIZE:].copy()
print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")

Train: 702, Val: 14, Test: 14


## Preprocessing Strategy

Minimal preprocessing — tree models handle raw features. Features created next.

In [9]:
df_full = df.copy().sort_values('date').reset_index(drop=True)
print('Data ready.')

Data ready.


## Feature Engineering

In [10]:
def create_features(data):
    d = data.copy()
    d['dow'] = d['date'].dt.dayofweek
    d['month'] = d['date'].dt.month
    d['day'] = d['date'].dt.day
    d['week_of_year'] = d['date'].dt.isocalendar().week.astype(int)
    for lag in [1, 7, 14, 28]:
        d[f'lag_{lag}'] = d[TARGET].shift(lag)
    for w in [7, 14, 28]:
        d[f'rmean_{w}'] = d[TARGET].shift(1).rolling(w).mean()
        d[f'rstd_{w}'] = d[TARGET].shift(1).rolling(w).std()
    return d

df_feat = create_features(df_full).dropna().reset_index(drop=True)
feature_cols = [c for c in df_feat.columns if c not in ['date', TARGET]]
print(f"Features: {len(feature_cols)} columns, {len(df_feat)} rows")

Features: 14 columns, 702 rows


## Baseline Model — Naive Forecasts

In [11]:
def calc_metrics(y_true, y_pred, name=""):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / np.maximum(y_true, 1))) * 100
    print(f"{name:30s} | MAE: {mae:10.1f} | RMSE: {rmse:10.1f} | MAPE: {mape:5.2f}%")
    return {'model': name, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

results = []
naive_pred = np.full(TEST_SIZE, train[TARGET].iloc[-1])
results.append(calc_metrics(test[TARGET].values, naive_pred, "Naive (Last Value)"))

src = df_full[TARGET].values
ti = len(df_full) - TEST_SIZE
sn_pred = src[ti - SEASON_LENGTH:ti - SEASON_LENGTH + TEST_SIZE]
results.append(calc_metrics(test[TARGET].values, sn_pred, f"Seasonal Naive ({SEASON_LENGTH})"))

Naive (Last Value)             | MAE:       17.1 | RMSE:       18.4 | MAPE:  9.40%
Seasonal Naive (7)             | MAE:        9.0 | RMSE:       10.6 | MAPE:  5.01%


## LazyPredict Benchmark (Lag-Feature Tabular)

In [12]:
from lazypredict.Supervised import LazyRegressor

ct = len(df_feat) - TEST_SIZE
cv = ct - VAL_SIZE
X_tr = df_feat.iloc[:cv][feature_cols]
y_tr = df_feat.iloc[:cv][TARGET]
X_va = df_feat.iloc[cv:ct][feature_cols]
y_va = df_feat.iloc[cv:ct][TARGET]

reg = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric=None, predictions=True)
lp_models, lp_preds = reg.fit(X_tr, X_va, y_tr, y_va)
print("\nLazyPredict Top 10:")
print(lp_models.head(10).to_string())


LazyPredict Top 10:
                          Adjusted R-Squared   R-Squared        RMSE  Time Taken
Model                                                                           
KernelRidge                      3647.469303 -279.497639  146.852725    0.014772
GaussianProcessRegressor         1609.313321 -122.716409   97.528391    0.034518
QuantileRegressor                  65.390314   -3.953101   19.514436    0.043047
DummyRegressor                     61.892569   -3.684044   18.977014    0.005996
MLPRegressor                       28.731966   -1.133228   12.806665    0.242397
NuSVR                              17.256732   -0.250518    9.805333    0.016823
SVR                                15.344850   -0.103450    9.210723    0.021709
KNeighborsRegressor                13.462379    0.041355    8.585117    0.007029
TweedieRegressor                   13.196496    0.061808    8.493042    0.008319
GammaRegressor                     12.822451    0.090581    8.361795    2.534902


## FLAML AutoML Run

In [13]:
from flaml import AutoML

automl = AutoML()
automl.fit(
    X_train=X_tr, y_train=y_tr,
    task='regression', time_budget=30, metric='mae',
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
print(f"FLAML best: {automl.best_estimator}")
flaml_val = automl.predict(X_va)
results.append(calc_metrics(y_va.values, flaml_val, f"FLAML ({automl.best_estimator})"))

X_te = df_feat.iloc[ct:][feature_cols]
y_te = df_feat.iloc[ct:][TARGET]
flaml_test = automl.predict(X_te)
results.append(calc_metrics(y_te.values, flaml_test, f"FLAML Test ({automl.best_estimator})"))

FLAML best: lgbm
FLAML (lgbm)                   | MAE:        4.5 | RMSE:        5.2 | MAPE:  2.80%
FLAML Test (lgbm)              | MAE:        7.4 | RMSE:        8.3 | MAPE:  4.07%


## StatsForecast — Statistical Forecasting

In [14]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, SeasonalNaive as SFSeasonalNaive

sf_df = df_full[['date', TARGET]].copy()
sf_df.columns = ['ds', 'y']
sf_df['unique_id'] = 'series1'
sf_df = sf_df[['unique_id', 'ds', 'y']]

sf_train = sf_df.iloc[:-TEST_SIZE]
sf = StatsForecast(
    models=[AutoARIMA(season_length=SEASON_LENGTH), AutoETS(season_length=SEASON_LENGTH),
            SFSeasonalNaive(season_length=SEASON_LENGTH)],
    freq=FREQ, n_jobs=1
)
sf.fit(sf_train)
sf_preds = sf.predict(h=TEST_SIZE).reset_index()

y_test_sf = test[TARGET].values
for col in ['AutoARIMA', 'AutoETS', 'SeasonalNaive']:
    if col in sf_preds.columns:
        results.append(calc_metrics(y_test_sf, sf_preds[col].values, f"SF {col}"))
print("StatsForecast complete.")

SF AutoARIMA                   | MAE:        6.5 | RMSE:        7.3 | MAPE:  3.62%
SF AutoETS                     | MAE:        6.7 | RMSE:        7.4 | MAPE:  3.70%
SF SeasonalNaive               | MAE:       11.9 | RMSE:       15.1 | MAPE:  6.52%
StatsForecast complete.


## Top 3 Model Selection

In [15]:
results_df = pd.DataFrame(results).sort_values('MAE').reset_index(drop=True)
print("All Results:")
print(results_df.to_string(index=False))
print("\nTop 3:")
print(results_df.head(3).to_string(index=False))

All Results:
             model       MAE      RMSE     MAPE
      FLAML (lgbm)  4.505159  5.239236 2.799944
      SF AutoARIMA  6.503034  7.323626 3.618620
        SF AutoETS  6.668581  7.429466 3.704882
 FLAML Test (lgbm)  7.423259  8.336292 4.073784
Seasonal Naive (7)  9.041429 10.615232 5.006788
  SF SeasonalNaive 11.862857 15.080351 6.517618
Naive (Last Value) 17.092857 18.410215 9.396834

Top 3:
       model      MAE     RMSE     MAPE
FLAML (lgbm) 4.505159 5.239236 2.799944
SF AutoARIMA 6.503034 7.323626 3.618620
  SF AutoETS 6.668581 7.429466 3.704882


## Final Training and Evaluation of Top 3

In [16]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(test['date'].values, test[TARGET].values, 'ko-', label='Actual', ms=4)
ax.plot(test['date'].values, flaml_test, 's--', label=f'FLAML ({automl.best_estimator})', ms=4)
for col in ['AutoARIMA', 'AutoETS']:
    if col in sf_preds.columns:
        ax.plot(test['date'].values[:len(sf_preds)], sf_preds[col].values, 'o--', label=f'SF {col}', ms=4)
ax.set_title('Forecast vs Actual — Test Set')
ax.legend()
plt.tight_layout()
plt.savefig('forecast_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

## Error Analysis

In [17]:
errors = test[TARGET].values - flaml_test
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(len(errors)), errors, alpha=0.7)
axes[0].set_title('Residuals (Actual - FLAML)')
axes[0].axhline(y=0, color='r', linestyle='--')
axes[1].hist(errors, bins=10, edgecolor='black', alpha=0.7)
axes[1].set_title('Residual Distribution')
plt.tight_layout()
plt.savefig('error_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"Mean residual: {np.mean(errors):.2f}, Std: {np.std(errors):.2f}")

Mean residual: 6.24, Std: 5.53


## Interpretation and Business Insight

- Stock prices follow an approximate random walk with upward drift
- The naive forecast (last value) is extremely hard to beat consistently
- Earnings jumps create discrete price level shifts
- Volatility regimes affect prediction intervals but not point forecasts much
- Directional accuracy of ~50% is expected for efficient markets

## Limitations

1. Synthetic — real stocks depend on fundamentals, sentiment, market microstructure
2. No fundamental data (earnings, revenue, P/E ratios)
3. No market data (volume, order book, sector indices)
4. Geometric Brownian Motion is a simplification — real returns have fat tails
5. No transaction costs or market impact modeling

## How to Improve This Project

1. Work with returns instead of levels for stationarity
2. Add fundamental features (earnings estimates, analyst revisions)
3. Include market microstructure data (volume, spreads)
4. Use proper walk-forward validation with retraining
5. Focus on volatility forecasting rather than price levels

## Production Considerations

- Stock price forecasting is inherently unreliable for trading
- Focus on risk estimation (VaR, Expected Shortfall) instead
- Proper backtesting with transaction costs is mandatory
- Regulatory compliance (model risk management) required

## Common Mistakes

1. Believing you can consistently beat the naive forecast on stock prices
2. Using RMSE on levels — it rewards models that memorize the last price
3. Not testing for lookahead bias and data leakage
4. Ignoring transaction costs when evaluating trading signals
5. Overfitting to in-sample patterns that don't persist

## Mini Challenge / Exercises

1. Compare RMSE of naive vs all models — how large is the gap?
2. Convert to returns and test for autocorrelation
3. Build a directional forecast and measure accuracy vs 50%
4. Identify earnings jump days from the return series

## Final Summary / Key Takeaways

1. Stock prices are near-random-walk — the hardest forecasting target
2. The naive forecast is surprisingly competitive on price levels
3. Returns are more appropriate than levels for modeling
4. Volatility is more forecastable than direction
5. Proper evaluation methodology matters more than model complexity

In [18]:
import json
metrics = {
    'project': 'Stock Close Price Forecasting',
    'best_model': results_df.iloc[0]['model'],
    'best_MAE': float(results_df.iloc[0]['MAE']),
    'best_RMSE': float(results_df.iloc[0]['RMSE']),
    'best_MAPE': float(results_df.iloc[0]['MAPE']),
    'all_results': results_df.to_dict('records')
}
with open('metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("Metrics saved.")
print("\n=== Stock Close Price Forecasting — Complete ===")

Metrics saved.

=== Stock Close Price Forecasting — Complete ===
